In [3]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3

In [4]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet) :
    print(name[0])
    print(name[1])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet) :
    print(name[0])
    print(name[1])
    print("============")

In [5]:
# pipeline configuring

geo      = utl.randomGeo(p=0.7)
crop     = utl.volume_crop((128 , 128 , 128))
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 128 , 128 , 128 , 3] ,
    labelShape=[None , 128 , 128 , 128 , 1]
)

windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=0.7 ,
    p_ww=0.7
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64])
def rimg(imgPath , labelPath) :
    return utl.read_img(imgPath , labelPath)
def read_img(img , label) :
    imglbl = rimg(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    img , label = geo.flip(
        img , 
        label
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



I0000 00:00:1781532806.124248   31192 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1781532806.248560   31192 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1781532806.255318   31192 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1

In [10]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    .cache("myCacheTrain")
    .shuffle(buffer_size=120)
    
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )

    .batch(batch_size=4)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTestSet , labelTestSet))

    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )
    .map(
        utl.channelize
    )
    .cache("myCacheValid")
    .batch(batch_size=2)
    .map(
        windower.apply_default , 
        num_parallel_calls=4
    )
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
for data in dataloaderValid.take(5) :
    print("image shape :" , data[0].shape)
    print("label shape :" , data[1].shape)

In [11]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4
)

# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze34_border , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

In [ ]:
print(model.summary())

In [12]:
# compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice
    ] ,
)

In [ ]:
logger = keras.callbacks.CSVLogger("tuning_logs/shuffleOpt_randomOpt_B4SH120.csv")

# model training
history = model.fit(
    x = dataloaderTrain ,
    epochs=50 ,
    validation_data = dataloaderValid ,
    callbacks=[
        logger
    ]

)

Epoch 1/50


I0000 00:00:1781532862.538352   31357 service.cc:153] XLA service 0x7e9594078360 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781532862.538419   31357 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1781532863.574277   31357 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781532869.076519   31357 cuda_dnn.cc:461] Loaded cuDNN version 92300
E0000 00:00:1781532880.432321   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781532894.208128   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:

94/95 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - dice: 1.4825e-04 - loss: 0.2587 - v__recall: 8.5208

E0000 00:00:1781533001.321034   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781533003.381670   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781533008.743930   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1781533016.704879   31357 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


95/95 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - dice: 1.4615e-04 - loss: 0.2578 - v__recall: 8.3846 - val_dice: 0.0000e+00 - val_loss: 0.2104 - val_v__recall: 0.0000e+00
Epoch 2/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 62s 612ms/step - dice: 0.0489 - loss: 0.1991 - v__recall: 5.3465 - val_dice: 0.0000e+00 - val_loss: 0.2005 - val_v__recall: 0.0000e+00
Epoch 3/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 60s 592ms/step - dice: 0.2378 - loss: 0.1736 - v__recall: 34.5166 - val_dice: 0.0000e+00 - val_loss: 0.2052 - val_v__recall: 0.0000e+00
Epoch 4/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 63s 617ms/step - dice: 0.3000 - loss: 0.1389 - v__recall: 36.4849 - val_dice: 0.0000e+00 - val_loss: 0.2039 - val_v__recall: 0.0000e+00
Epoch 5/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 63s 617ms/step - dice: 0.5895 - loss: 0.0781 - v__recall: 65.3046 - val_dice: 0.0000e+00 - val_loss: 0.2043 - val_v__recall: 0.0000e+00
Epoch 6/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 62s 612ms/step - dice: 0.5497 - loss: 0.0890 - v__recall: 61.7608 - val_dice: 0.0958 - val_loss: 0.1755

In [9]:
model.save("models/NoShuffle_FullRandom.keras")

In [ ]:
model = keras.models.load_model("models/NoShuffle_FullRandom.keras")